In [20]:
#Import models and libraries
from DT.DecisionTree import DecisionTree
from sklearn.tree import DecisionTreeClassifier
from GNB.gaussian_naive_bayes import GNB
from LogisticRegresssion.LogisticRegression import LogReg
import keras
from SVM.linear_svm import LinearSVMScartch
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report , f1_score
from sklearn.model_selection import train_test_split



In [21]:
#Download Data
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()


In [22]:
y_train_bin = (y_train == 6).astype(int)
y_test_bin  = (y_test == 6).astype(int)


In [23]:
# Flatten
# X_train = X_train.reshape(X_train.shape[0], -1)
# X_test = X_test.reshape(X_test.shape[0], -1)

In [24]:
# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

In [25]:
def compute_pixel_variance(X):
    mean = np.mean(X, axis=0)
    return np.mean((X - mean) ** 2, axis=0)

def variance_threshold(X, threshold=1e-4):
    variances = compute_pixel_variance(X)
    mask = variances > threshold
    return X[:, mask], mask

X_train_clean, mask = variance_threshold(X_train)
X_test_clean = X_test[:, mask]

print("After variance filtering:", X_train_clean.shape)


After variance filtering: (60000, 623)


In [26]:
import numpy as np
from skimage.feature import hog

def extract_hog_features(X):
    features = []

    for img in X:
        img_2d = img.reshape(28, 28)

        hog_features = hog(
            img_2d,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            feature_vector=True
        )

        features.append(hog_features)

    return np.array(features)

In [27]:
X_train_HOG = extract_hog_features(X_train)
X_test_HOG = extract_hog_features(X_test)

In [28]:
pca = PCA(n_components=50)  

X_train_hog_pca = pca.fit_transform(X_train_HOG)
X_test_hog_pca = pca.transform(X_test_HOG)

In [29]:
print(X_test_hog_pca.shape)

(10000, 50)


In [30]:
#Decision Tree results
dt = DecisionTree(maxDepth=12 , minSampleLeafs=5 , minSamplesSplit=10 , criterion='entropy' ,maxFeatures= 'sqrt' , classWeights={0:1 , 1:10} )


dt.fit(X_train_hog_pca , y_train_bin)
predictions = dt.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin, predictions,
    target_names=["Not 6", "Is 6"]
))



              precision    recall  f1-score   support

       Not 6       1.00      0.96      0.98      9042
        Is 6       0.74      0.97      0.84       958

    accuracy                           0.96     10000
   macro avg       0.87      0.97      0.91     10000
weighted avg       0.97      0.96      0.97     10000



In [31]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_hog_pca, y_train_bin,
    test_size=0.2,
    stratify=y_train_bin,
    random_state=42
)

# Train
gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

# Tune weights
best_weight = None
best_score = -1

for w in [1, 1.5, 2, 3, 4, 5, 7, 10]:
    class_weights = {0: 1.0, 1: w}
    preds = gnb.predict(X_val, class_weights=class_weights)
    score = f1_score(y_val, preds, pos_label=1)

    print(f"Weight {w} → F1: {score:.4f}")

    if score > best_score:
        best_score = score
        best_weight = w

print("\n🔥 Best weight:", best_weight)

# Test
best_weights = {0: 1.0, 1: best_weight}
predictions = gnb.predict(X_test_hog_pca, class_weights=best_weights)

print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

Weight 1 → F1: 0.9329
Weight 1.5 → F1: 0.9402
Weight 2 → F1: 0.9436
Weight 3 → F1: 0.9480
Weight 4 → F1: 0.9527
Weight 5 → F1: 0.9550
Weight 7 → F1: 0.9557
Weight 10 → F1: 0.9567

🔥 Best weight: 10
              precision    recall  f1-score   support

       Not 6       0.99      1.00      1.00      9042
        Is 6       0.98      0.93      0.96       958

    accuracy                           0.99     10000
   macro avg       0.99      0.97      0.98     10000
weighted avg       0.99      0.99      0.99     10000



In [32]:
lg = LogReg(max_iterations=1000 , learning_rate=0.1 , threshold=0.5)
lg.fit(X_train_hog_pca , y_train_bin)
predictions = lg.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

Iteration 0, Loss: 0.6931471805599453
Iteration 10, Loss: 0.520499733917744
Iteration 20, Loss: 0.4146008649303269
Iteration 30, Loss: 0.3451502103344921
Iteration 40, Loss: 0.29673142376773565
Iteration 50, Loss: 0.2612525854544718
Iteration 60, Loss: 0.23420157119945187
Iteration 70, Loss: 0.2129085923865931
Iteration 80, Loss: 0.19570946478087442
Iteration 90, Loss: 0.18151947937902477
Iteration 100, Loss: 0.1696041385060571
Iteration 110, Loss: 0.15944954773639763
Iteration 120, Loss: 0.15068587199477265
Iteration 130, Loss: 0.14304035804028115
Iteration 140, Loss: 0.13630750828211927
Iteration 150, Loss: 0.13032956679127275
Iteration 160, Loss: 0.1249834066295985
Iteration 170, Loss: 0.120171505980629
Iteration 180, Loss: 0.11581560389419493
Iteration 190, Loss: 0.11185215326446134
Iteration 200, Loss: 0.10822900475337413
Iteration 210, Loss: 0.10490295000228117
Iteration 220, Loss: 0.10183787517904776
Iteration 230, Loss: 0.09900335495646914
Iteration 240, Loss: 0.096373568962352

In [33]:
y_train_bin = np.where(y_train == 6, 1, -1)
y_test_bin  = np.where(y_test == 6, 1, -1)

In [34]:
svm = LinearSVMScartch(
    C=1.0,
    learning_rate=0.0001,
    n_epochs=100,
    batch_size=128,
    use_class_weights=True,
    random_state=42
)
svm.fit(X_train_hog_pca , y_train_bin)
predictions = svm.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

/Users/amirtamer/CAIE/Sem 6/ML/Project/SVM/linear_svm.py:32: RuntimeWarning: divide by zero encountered in scalar divide
  weight_pos = n_samples / (2.0 * n_pos)


IndexError: index 51012 is out of bounds for axis 0 with size 48000